# PySpark Extension Types

The types that are used by the AWS Glue PySpark extensions.

## Type Inheritance Hierarchy

```
DataType                        ← base class for everything
├── AtomicType(DataType)        ← base for all simple/primitive types
│   ├── StringType
│   ├── IntegerType
│   ├── DoubleType
│   ├── DecimalType(AtomicType) ← extends AtomicType
│   └── EnumType(AtomicType)   ← extends AtomicType
├── ArrayType(DataType)         ← extends DataType directly (holds a collection)
├── MapType(DataType)           ← extends DataType directly (holds key-value pairs)
├── StructType(DataType)        ← extends DataType directly (holds multiple fields)
└── ChoiceType(DataType)        ← extends DataType directly (holds mixed types)
```

**Rule of thumb:**
- Extends `AtomicType` → holds a **single primitive value** (string, number, enum etc)
- Extends `DataType` directly → holds **complex/nested/multiple values** (array, map, struct etc)

In [34]:
spark.stop()

In [1]:
from pyspark.context import SparkContext
from awsglue.context import GlueContext

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

# sc._jsc.hadoopConfiguration().set('fs.s3a.endpoint', 'http://minio:9000')
# sc._jsc.hadoopConfiguration().set('fs.s3a.access.key', 'minioadmin')
# sc._jsc.hadoopConfiguration().set('fs.s3a.secret.key', 'minioadmin')
# sc._jsc.hadoopConfiguration().set('fs.s3a.path.style.access', 'true')
# sc._jsc.hadoopConfiguration().set('fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
# sc._jsc.hadoopConfiguration().set('fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
# sc._jsc.hadoopConfiguration().set('fs.s3a.connection.ssl.enabled', 'false')
print('session ready')

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/home/glue_user/spark/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/home/glue_user/spark/jars/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/home/glue_user/aws-glue-libs/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/home/glue_user/aws-glue-libs/jars/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/glue_user/spark/python/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getO

session ready


---
## DataType

The base class for all AWS Glue types. You won't use this directly — all other types inherit from it.

**`__init__(properties={})`**
+ `properties` – Properties of the data type (optional).

**`typeName(cls)`**

Returns the type name (class name with "Type" removed from the end).

**`jsonValue()`**

Returns a JSON representation of the type:
```json
{
  "dataType": "string",
  "properties": {}
}
```

In [8]:
from awsglue.gluetypes import StringType, IntegerType

print(StringType().jsonValue())
# {"dataType": "string", "properties": {}}

print(IntegerType().jsonValue())
# {"dataType": "integer", "properties": {}}

{'dataType': 'string', 'properties': {}}
{'dataType': 'int', 'properties': {}}


---
## AtomicType and simple derivatives

Inherits from `DataType`. Base class for all simple/primitive types in Glue.

These are used when defining schemas for DynamicFrames using `StructType` + `Field`.

The following types are simple derivatives of `AtomicType`:
+ `BinaryType` – Binary data.
+ `BooleanType` – Boolean values.
+ `ByteType` – A byte value.
+ `DateType` – A datetime value.
+ `DoubleType` – A floating-point double value.
+ `IntegerType` – An integer value.
+ `LongType` – A long integer value.
+ `NullType` – A null value.
+ `ShortType` – A short integer value.
+ `StringType` – A text string.
+ `TimestampType` – A timestamp value.
+ `UnknownType` – A value of unidentified type.

In [9]:
from awsglue.gluetypes import *

# Read DynamicFrame first
dyf = glueContext.create_dynamic_frame.from_options(
    connection_type='s3',
    connection_options={'paths': ['s3a://glue-spark-etl-example-source/raw/orders/orders6.csv']},
    format='csv',
    format_options={'withHeader': True}
)

# Glue gluetypes are used to INSPECT the schema of an existing DynamicFrame
# dyf.schema() returns a Glue StructType object
glue_schema = dyf.schema()
# print(type(glue_schema))         # <class 'awsglue.gluetypes.StructType'>
# dyf.printSchema()
# print(glue_schema.jsonValue())   # full schema as JSON

print(glue_schema.fields)
# Each field in the schema is a Glue Field object with a Glue type
for field in glue_schema.fields:
    print(f'{field.name} -> {type(field.dataType).__name__}')


26/04/26 16:41:49 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


[Field(order_id, StringType({}), {}), Field(customer_name, StringType({}), {}), Field(product, StringType({}), {}), Field(amount, StringType({}), {}), Field(order_date, StringType({}), {})]
order_id -> StringType
customer_name -> StringType
product -> StringType
amount -> StringType
order_date -> StringType


---
## DecimalType(AtomicType)

Represents a decimal number with configurable precision and scale.

**`__init__(precision=10, scale=2, properties={})`**
+ `precision` – Total number of digits (default: 10).
+ `scale` – Digits to the right of the decimal point (default: 2).

In [10]:
from awsglue.gluetypes import DecimalType, StringType, StructType, Field

# precision=10, scale=2 means up to 99999999.99
schema = StructType([
    Field('order_id', StringType()),
    Field('amount',   DecimalType(precision=10, scale=2))
])

print(DecimalType(10, 2).jsonValue())
# {'dataType': 'decimal', 'properties': {}, 'precision': 10, 'scale': 2}

{'dataType': 'decimal', 'properties': {}, 'precision': 10, 'scale': 2}


---
## EnumType(AtomicType)

Represents a column that can only hold a fixed set of values.

**`__init__(options)`**
+ `options` – A list of valid values.

In [11]:
from awsglue.gluetypes import EnumType, StringType, StructType, Field

schema = StructType([
    Field('order_id', StringType()),
    Field('gender',   EnumType(['M', 'F', 'Other'])),
    Field('status',   EnumType(['active', 'inactive', 'suspended']))
])

# Note: jsonValue() returns None for EnumType in Glue 4 — known incomplete implementation
print(EnumType(['M', 'F']).jsonValue())  # None

# EnumType is still usable in schema definitions
print(schema.hasField('gender'))   # True
print(type(schema.getField('gender').dataType).__name__)  # EnumType

None
True
EnumType


---
## ArrayType(DataType)

Represents a column that holds a list/array of values of the same type.

**`__init__(elementType=UnknownType(), properties={})`**
+ `elementType` – The type of each element in the array.

### Note: Type inference depends on source format

| Format | Type Inference |
|--------|---------------|
| CSV | Always `string` |
| JSON | Infers from values (`[...]` → array, `{...}` → struct, `123.45` → double etc) |
| Parquet | Reads from metadata |
| Avro | Reads from schema |

Sample JSON for testing:
```json
{"order_id": "ORD-001", "tags": ["electronics", "premium", "sale"]}
{"order_id": "ORD-002", "tags": ["clothing"]}
```

In [12]:
from awsglue.gluetypes import ArrayType, StringType, StructType, Field
from awsglue.dynamicframe import DynamicFrame

print(ArrayType(StringType()).jsonValue())

data = [
    {'order_id': 'ORD-001', 'tags': ['electronics', 'premium', 'sale']},
    {'order_id': 'ORD-002', 'tags': ['clothing']}
]
dyf = DynamicFrame.fromDF(spark.createDataFrame(data), glueContext, 'array_sample')
dyf.printSchema()
dyf.show()

{'dataType': 'array', 'properties': {}, 'elementType': {'dataType': 'string', 'properties': {}}}
root
|-- order_id: string
|-- tags: array
|    |-- element: string



{"order_id": "ORD-001", "tags": ["electronics", "premium", "sale"]}
{"order_id": "ORD-002", "tags": ["clothing"]}


## ChoiceType(DataType)

Represents a column that has **mixed/ambiguous types** — this is what Glue creates internally when it detects a column has both `string` and `double` values. This is the type that `resolveChoice()` is designed to fix.

**`__init__(choices=[], properties={})`**
+ `choices` – A list of possible types for this column.

**`add(new_choice)`** – Adds a new type to the choices list.

**`merge(new_choices)`** – Merges a list of new types with existing choices.

Sample JSON for testing (amount is sometimes a number, sometimes a string):
```json
{"order_id": "ORD-001", "amount": 132.63}
{"order_id": "ORD-002", "amount": "N/A"}
```

In [13]:
from awsglue.dynamicframe import DynamicFrame

data = [
    {'order_id': 'ORD-001', 'amount': 132.63},
    {'order_id': 'ORD-002', 'amount': 'N/A'}
]
dyf = DynamicFrame.fromDF(spark.createDataFrame(data), glueContext, 'choice_sample')
dyf.printSchema()

resolved = dyf.resolveChoice(specs=[('amount', 'cast:double')])
resolved.printSchema()
resolved.show()

TypeError: field amount: Can not merge type <class 'pyspark.sql.types.DoubleType'> and <class 'pyspark.sql.types.StringType'>

---
## MapType(DataType)

Represents a column that holds key-value pairs (like a Python dict).

**`__init__(valueType=UnknownType, properties={})`**
+ `valueType` – The type of the values in the map (keys are always strings).

Sample JSON for testing:
```json
{"order_id": "ORD-001", "metadata": {"source": "web", "campaign": "summer_sale"}}
{"order_id": "ORD-002", "metadata": {"source": "mobile"}}
```

In [14]:
from awsglue.gluetypes import MapType, StringType, StructType, Field
from awsglue.dynamicframe import DynamicFrame

print(MapType(StringType()).jsonValue())

data = [
    {'order_id': 'ORD-001', 'metadata': {'source': 'web', 'campaign': 'summer_sale'}},
    {'order_id': 'ORD-002', 'metadata': {'source': 'mobile'}}
]
dyf = DynamicFrame.fromDF(spark.createDataFrame(data), glueContext, 'map_sample')
dyf.printSchema()
dyf.show()

{'dataType': 'map', 'properties': {}, 'valueType': {'dataType': 'string', 'properties': {}}}
root
|-- metadata: map
|    |-- keyType: string
|    |-- valueType: string
|-- order_id: string

{"metadata": {"campaign":"summer_sale", "source":"web"}, "order_id": "ORD-001"}
{"metadata": {"source":"mobile"}, "order_id": "ORD-002"}


---
## Field(Object)

Represents a single column in a `StructType` schema. Always used together with `StructType`.

**`__init__(name, dataType, properties={})`**
+ `name` – Column name.
+ `dataType` – Any type derived from `DataType`.

In [15]:
from awsglue.gluetypes import Field, StringType, DoubleType

order_id_field = Field('order_id', StringType())
amount_field   = Field('amount',   DoubleType())

print(order_id_field.name)
print(order_id_field.dataType)

order_id
StringType({})


---
## StructType(DataType)

Defines the full schema of a DynamicFrame as a collection of `Field` objects. This is the Glue equivalent of Spark's `StructType`.

**`__init__(fields=[], properties={})`**
+ `fields` – List of `Field` objects.

**`add(field)`** – Add a new field to the schema.

**`hasField(field)`** – Returns `True` if field exists. *(Glue only — not in Spark)*

**`getField(field)`** – Returns the field by name. *(Glue only — not in Spark)*

In [16]:
from awsglue.gluetypes import StructType, Field, StringType, DoubleType, DateType, BooleanType

schema = StructType([
    Field('order_id',      StringType()),
    Field('customer_name', StringType()),
    Field('amount',        DoubleType()),
    Field('order_date',    DateType()),
    Field('is_vip',        BooleanType())
])

# hasField — Glue only
print(schema.hasField('amount'))    # True
print(schema.hasField('quantity'))  # False

# getField — Glue only
field = schema.getField('amount')
print(field.name)                   # amount

# add a new field dynamically
schema.add(Field('processed_date', DateType()))
print(schema.hasField('processed_date'))  # True

True
False
amount
True


---
## EntityType(DataType)

`__init__(entity, base_type, properties)`

**Not yet implemented by AWS.** Reserved for future use.

---
## DataSource(object)

Represents a data source returned by `glueContext.getSource()`. Used to read data into a DynamicFrame.

**`setFormat(format, **options)`** – Set the format to read.

**`getFrame()`** – Returns a `DynamicFrame`.

In [36]:
# DataSource is what glueContext.getSource() returns internally
# Most common usage via create_dynamic_frame, but getSource gives more control

data_source = glueContext.getSource(
    connection_type='s3',
    connection_options={'paths': ['s3a://glue-spark-etl-example-source/raw/orders/orders6.csv']}
)

[print(i) for i in dir(data_source) if not i.startswith('_')]

data_source.setFormat('csv', withHeader=True)

dyf = data_source.getFrame()
# dyf.printSchema()
# dyf.show(3)

getFrame
getSampleFrame
name
setFormat


In [ ]:
# sc = SparkContext.getOrCreate()
# glueContext = GlueContext(sc)
# spark = glueContext.spark_session
print(type(glueContext))
print(type(sc))
print(type(spark))

<class 'awsglue.context.GlueContext'>
<class 'pyspark.context.SparkContext'>
<class 'pyspark.sql.session.SparkSession'>


In [26]:
data_catalog_source_orders= glueContext.create_dynamic_frame.from_catalog(
    database='ecommerce_db',
    table_name='orders'
)

# [print(i) for i in dir(data_catalog_source_orders) if not i.startswith('_')]

df1= data_catalog_source_orders.toDF()
df1.cache()

df1.count()
# dyf2 = data_catalog_source_orders.repartition(2)
# dyf2.getNumPartitions()
# dyf3=


/home/glue_user/spark/python/pyspark/sql/dataframe.py:127: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


5000

In [31]:
df1.count()

5000

In [62]:
from awsglue.dynamicframe import DynamicFrame

dyf2= DynamicFrame.fromDF(df1, glueContext, "test")
from awsglue.dynamicframe import DynamicFrame

dyf2= DynamicFrame.fromDF(df1, glueContext, "test")
dyf2.count()

5000

In [64]:
glueContext.write_dynamic_frame.from_catalog(
    frame = dyf2,
    database = 'ecommerce_db',
    table_name = 'orders',
    transformation_ctx = 'write_to_catalog')

In [60]:
help(glueContext.write_data_frame.from_catalog)

Help on method from_catalog in module awsglue.dataframewriter:

from_catalog(frame, database=None, table_name=None, redshift_tmp_dir='', transformation_ctx='', additional_options={}, catalog_id=None, **kwargs) method of awsglue.dataframewriter.DataFrameWriter instance
    Writes a DataFrame with the specified catalog name space and table name.



In [58]:
[print(m) for m in dir(glueContext) if m.startswith("w")]


write_data_frame
write_data_frame_from_catalog
write_dynamic_frame
write_dynamic_frame_from_catalog
write_dynamic_frame_from_jdbc_conf
write_dynamic_frame_from_options
write_from_jdbc_conf
write_from_options


[None, None, None, None, None, None, None, None]

In [76]:
datasource = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": ['s3://sanjay-de-bucket-2026/raw/orders/']},
    format="csv",
    format_options={"withHeader": True, "separator": ","}
)
datasource.show(5)

{"order_id": "ORD00001", "order_date": "2024-07-01", "customer_id": "CUST0212", "product_id": "P006", "product_name": "Jacket", "category": "Clothing", "quantity": "3", "unit_price": "80"}
{"order_id": "ORD00002", "order_date": "2024-05-03", "customer_id": "CUST0157", "product_id": "P004", "product_name": "T-Shirt", "category": "Clothing", "quantity": "1", "unit_price": "20"}
{"order_id": "ORD00003", "order_date": "2024-02-04", "customer_id": "CUST0109", "product_id": "P002", "product_name": "Phone", "category": "Electronics", "quantity": "2", "unit_price": "500"}
{"order_id": "ORD00004", "order_date": "2024-06-27", "customer_id": "CUST0393", "product_id": "P011", "product_name": "Chair", "category": "Home", "quantity": "3", "unit_price": "120"}
{"order_id": "ORD00005", "order_date": "2024-04-19", "customer_id": "CUST0075", "product_id": "P006", "product_name": "Jacket", "category": "Clothing", "quantity": "5", "unit_price": "80"}


In [54]:
glueContext.write_dynamic_frame.from_options(
    frame = datasource,
    connection_type = "s3",
    connection_options = {"path": "s3://sanjay-de-bucket-2026/processed/orders_parquet/"}, 
    format = "parquet"
)

In [53]:
[print(i) for i in dir(glueContext) if  i.startswith('w')]

write_data_frame
write_data_frame_from_catalog
write_dynamic_frame
write_dynamic_frame_from_catalog
write_dynamic_frame_from_jdbc_conf
write_dynamic_frame_from_options
write_from_jdbc_conf
write_from_options


[None, None, None, None, None, None, None, None]

## This is a known issue — Spark SQL doesn't honor skip.header.line.count table property from Glue Catalog, even though Athena does.

https://stackoverflow.com/questions/50092022/how-to-ignore-headers-in-pyspark-when-using-athena-and-aws-glue-data-catalog

In [73]:
import os
os.environ['AWS_PROFILE'] = 'default'
os.environ['AWS_SDK_LOAD_CONFIG'] = '1'
df3= spark.sql("select * from order_analytics_dev.order_daily_trends where order_date != 'order_date'")
print(df3.count())
df3.show(5)


365
+----------+----------+-------------+
|order_date|num_orders|daily_revenue|
+----------+----------+-------------+
|2024-01-01|        12|       2370.0|
|2024-01-02|         9|       1375.0|
|2024-01-03|        11|       5655.0|
|2024-01-04|        14|       2075.0|
|2024-01-05|         9|       3525.0|
+----------+----------+-------------+
only showing top 5 rows



---
## DataSink(object)

Represents a data sink returned by `glueContext.getSink()`. Used to write a DynamicFrame to a destination.

**`setFormat(format, **options)`** – Set the output format.

**`setAccumulableSize(size)`** – Set the accumulable size in bytes.

**`writeFrame(dynamic_frame, info="")`** – Write a single DynamicFrame to the sink.

**`write(dynamic_frame_or_dfc, info="")`** – Write a DynamicFrame or DynamicFrameCollection.

In [2]:
# DataSink is what glueContext.getSink() returns
sink = glueContext.getSink(
    connection_type='s3',
    path='s3a://glue-spark-etl-example-target/silver/orders/',
    partitionKeys=['order_date']
)
sink.setFormat('parquet', compression='snappy')

# writeFrame — write a single DynamicFrame
sink.writeFrame(dyf)
print('write completed')

Py4JJavaError: An error occurred while calling o52.setFormat.
: java.util.NoSuchElementException: key not found: useGlueParquetWriter
	at scala.collection.immutable.Map$Map1.apply(Map.scala:163)
	at com.amazonaws.services.glue.sinks.HadoopDataSink.setFormat(HadoopDataSink.scala:353)
	at sun.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at sun.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:62)
	at sun.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.lang.reflect.Method.invoke(Method.java:498)
	at py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244)
	at py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:357)
	at py4j.Gateway.invoke(Gateway.java:282)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)
	at py4j.commands.CallCommand.execute(CallCommand.java:79)
	at py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:182)
	at py4j.ClientServerConnection.run(